In [ ]:
import pandas as pd
import numpy as np

# =========================================
# LOAD FILE
# =========================================
file_path = "Sample of coupled data.xlsx"   # change path if needed

df = pd.read_excel(file_path, sheet_name="Sheet1", header=[0,1])

# =========================================
# FLATTEN MULTI-LEVEL HEADERS
# =========================================
df.columns = [' | '.join([str(c).strip() for c in col if pd.notna(c)])
              for col in df.columns]

# =========================================
# IDENTIFY PART / MATERIAL COLUMN
# =========================================
material_col = [col for col in df.columns if "Part" in col or "Material" in col][0]

materials = df[material_col]

# =========================================
# SELECT ONLY DEVIATION COLUMNS
# =========================================
deviation_cols = [col for col in df.columns if "Deviation" in col]

deviation_data = df[deviation_cols]

# =========================================
# ROW-WISE STATISTICS (same logic as yours)
# =========================================
mean_dev = deviation_data.mean(axis=1)
std_dev = deviation_data.std(axis=1, ddof=1)
count = deviation_data.count(axis=1)

std_error = std_dev / np.sqrt(count)

# 🔹 90% Confidence Interval
z = 1.645

lower_ci = mean_dev - z * std_error
upper_ci = mean_dev + z * std_error

# =========================================
# RESULT
# =========================================
result = pd.DataFrame({
    'Material': materials,
    'Mean_Deviation': mean_dev,
    'Std_Dev': std_dev,
    'Lower_90_CI': lower_ci,
    'Upper_90_CI': upper_ci
})

print(result)

# Optional save
result.to_excel("Deviation_Confidence_Output.xlsx", index=False)

In [ ]:
import pandas as pd
import numpy as np

file_path = "Coupled data Updated.xlsx"

df = pd.read_excel(file_path, sheet_name="Sheet2")

# Identify columns
part_col = "Part"
actual_cols = [col for col in df.columns if "Actual" in col]

# Extract
parts = df[part_col]
actual_data = df[actual_cols]

# Statistics
mean_actual = actual_data.mean(axis=1)
std_actual = actual_data.std(axis=1, ddof=1)

# 90% CI
z = 1.645

lower_actual = mean_actual - z * std_actual
upper_actual = mean_actual + z * std_actual

# Result
result = pd.DataFrame({
    "Part": parts,
    "Mean_Actual": mean_actual,
    "Std_Actual": std_actual,
    "Lower_Actual_90": lower_actual,
    "Upper_Actual_90": upper_actual
})

print(result)

result.to_excel("Actual_90CI_Output.xlsx", index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================
# LOAD FILE
# ============================

file_path = "Coupled data Updated.xlsx"

df = pd.read_excel(file_path, sheet_name="Sheet2")

# ============================
# FIND ACTUAL COLUMNS
# ============================

actual_cols = [col for col in df.columns if "Actual" in col]

print("Actual columns detected:")
print(actual_cols)

# ============================
# SELECT PART (change index if needed)
# ============================

row_index = 0   # first part — change to analyze another part

values = df.loc[row_index, actual_cols].dropna()

print("\nValues used for histogram:")
print(values)

# ============================
# PLOT HISTOGRAM
# ============================

plt.figure(figsize=(8,5))
plt.hist(values, bins=10)
plt.title(f"Actual Distribution — Part: {df.loc[row_index, 'Part']}")
plt.xlabel("Actual Values")
plt.ylabel("Frequency")
plt.show()

In [ ]:
import pandas as pd

bom_file = r"D:/Tushar/main_with_subs_only.xlsx"
coupled_file = r"D:/Part Production Report/Coupled data Updated.xlsx"

# Read files
bom_df = pd.read_excel(bom_file)
comp_df = pd.read_excel(coupled_file, sheet_name="Sheet2")

bom_df.columns = bom_df.columns.str.strip()
comp_df.columns = comp_df.columns.str.strip()

def normalize(series):
    return series.astype(str).str.strip().str.upper()

# Normalize keys
bom_df['Sub_Label'] = normalize(bom_df['Sub_Label'])
bom_df['Main_Label'] = normalize(bom_df['Main_Label'])
comp_df['Part'] = normalize(comp_df['Part'])

# Prepare output
child_list = bom_df['Main_Label'].unique()
output_df = pd.DataFrame({'Material': child_list})

# ===============================
# LOOP PLANNING COLUMNS
# ===============================

for col in comp_df.columns:

    if col == 'Part':
        continue

    # choose columns to explode
    if not any(keyword in col for keyword in ["Actual", "Tentative", "Indent"]):
        continue

    print(f"Processing: {col}")

    comp_df['Demand'] = pd.to_numeric(comp_df[col], errors='coerce').fillna(0)
    lookup = dict(zip(comp_df['Part'], comp_df['Demand']))

    # SAME BOM LOGIC
    results = []
    current_child = None
    running_total = 0

    for _, row in bom_df.iterrows():

        child = row['Main_Label']
        switch = row['Sub_Label']
        usage = pd.to_numeric(row['Sub_Count'], errors='coerce') or 0

        daily_switch = lookup.get(switch, 0)
        contribution = daily_switch * usage

        if current_child is None:
            current_child = child

        if child != current_child:
            results.append((current_child, running_total))
            current_child = child
            running_total = 0

        running_total += contribution

    if current_child is not None:
        results.append((current_child, running_total))

    col_df = pd.DataFrame(results, columns=['Material', col])

    output_df = output_df.merge(col_df, on='Material', how='left')

output_df = output_df.fillna(0)

output_df.to_excel("Child_From_Coupled_Data.xlsx", index=False)

print("Saved output file.")

In [ ]:
import pandas as pd
import numpy as np

file_path = "Child_From_Coupled_Data.xlsx"

df = pd.read_excel(file_path)

# ============================
# Find indent columns
# ============================

indent_cols = [col for col in df.columns if "Indent" in col]

print("Indent columns detected:")
print(indent_cols)

# ============================
# Extract data
# ============================

indent_data = df[indent_cols]

# ============================
# Statistics
# ============================

mean_indent = indent_data.mean(axis=1)
std_indent = indent_data.std(axis=1, ddof=1)

z = 1.645  # 90% CI

lower_ci = mean_indent - z * std_indent
upper_ci = mean_indent + z * std_indent

# ============================
# Result
# ============================

result = pd.DataFrame({
    "Material": df["Material"],
    "Mean_Indent": mean_indent,
    "Std_Indent": std_indent,
    "Lower_Indent_90": lower_ci,
    "Upper_Indent_90": upper_ci
})

print(result)

result.to_excel("Child_Indent_CI.xlsx", index=False)

In [ ]:
import pandas as pd
import numpy as np

file_path = "Child_From_Coupled_Data.xlsx"

df = pd.read_excel(file_path)

# ============================
# Detect columns
# ============================

indent_cols = [col for col in df.columns if "Indent" in col]
tentative_cols = [col for col in df.columns if "Tentative" in col]
actual_cols = [col for col in df.columns if "Actual" in col]

indent_data = df[indent_cols]
tentative_data = df[tentative_cols]
actual_data = df[actual_cols]

# ============================
# ---- INDENT STATS ----
# ============================

mean_indent = indent_data.mean(axis=1)
std_indent = indent_data.std(axis=1, ddof=1)

z = 1.645  # 90% CI

lower_indent = mean_indent - z * std_indent
upper_indent = mean_indent + z * std_indent

indent_result = pd.DataFrame({
    "Material": df["Material"],
    "Mean_Indent": mean_indent,
    "Std_Indent": std_indent,
    "Lower_Indent_90": lower_indent,
    "Upper_Indent_90": upper_indent
})

# ============================
# ---- DEVIATIONS ----
# ============================

dev_actual_indent = actual_data.values - indent_data.values
dev_actual_tent = actual_data.values - tentative_data.values

dev_ai_df = pd.DataFrame(dev_actual_indent)
dev_at_df = pd.DataFrame(dev_actual_tent)

mean_ai = dev_ai_df.mean(axis=1)
std_ai = dev_ai_df.std(axis=1, ddof=1)

mean_at = dev_at_df.mean(axis=1)
std_at = dev_at_df.std(axis=1, ddof=1)

lower_ai = mean_ai - z * std_ai
upper_ai = mean_ai + z * std_ai

lower_at = mean_at - z * std_at
upper_at = mean_at + z * std_at

deviation_result = pd.DataFrame({
    "Material": df["Material"],

    "Mean_Dev_Actual_Indent": mean_ai,
    "Std_Dev_Actual_Indent": std_ai,
    "Lower_AI_90": lower_ai,
    "Upper_AI_90": upper_ai,

    "Mean_Dev_Actual_Tentative": mean_at,
    "Std_Dev_Actual_Tentative": std_at,
    "Lower_AT_90": lower_at,
    "Upper_AT_90": upper_at
})

# ============================
# SAVE TO SINGLE FILE
# ============================

output_file = "Child_Full_Analysis.xlsx"

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    indent_result.to_excel(writer, sheet_name="Indent_Stats", index=False)
    deviation_result.to_excel(writer, sheet_name="Deviation_Stats", index=False)

print("Saved combined analysis to:", output_file)

In [ ]:
import pandas as pd
import numpy as np

file_path = "Child_From_Coupled_Data.xlsx"

df = pd.read_excel(file_path)

# Detect deviation columns (Actual - Tentative already computed)
deviation_cols = [col for col in df.columns if "Deviation" in col]

deviation_data = df[deviation_cols]

# Stats
mean_dev = deviation_data.mean(axis=1)
std_dev = deviation_data.std(axis=1, ddof=1)

z = 1.645  # 90%

lower_dev = mean_dev - z * std_dev
upper_dev = mean_dev + z * std_dev

# Suppose today's tentative demand column
today_tentative = df["19TH Feb Plan Tentative"]  # change as needed

lower_demand = today_tentative + lower_dev
upper_demand = today_tentative + upper_dev

result = pd.DataFrame({
    "Material": df["Material"],
    "Tentative": today_tentative,
    "Lower_Expected_Actual": lower_demand,
    "Upper_Expected_Actual": upper_demand
})

print(result)